In [8]:
import warnings

warnings.filterwarnings("ignore")

In [9]:
# =============================================================================
# CMIP6 Climatology Calculator
# Computes detrended monthly climatologies for multiple models and variables
# Optimized for NCAR Casper HPC (4 nodes, 32GB/node)
# =============================================================================

# --- Standard Library ---
import gc
import os

# --- Scientific Stack ---
import numpy as np
import xarray as xr
import xesmf as xe
from scipy.signal import detrend

# --- Dask ---
from dask.distributed import Client, LocalCluster

# --- CMIP6 Catalog ---
import intake

In [10]:
# =============================================================================
# CONFIGURATION
# =============================================================================

START_YEAR = "1991"
END_YEAR   = "2014"
DEPTH_LEVEL = 0          # 0 = surface; set to depth in metres for 3-D vars

# Output region of interest (lat/lon subset saved to disk)
LAT_MIN, LAT_MAX = -90, 90   # change to your region, e.g. 45, 50
LON_MIN, LON_MAX =   1, 361  # change to your region, e.g. 210, 220

# Output 1° grid
LAT_OUT = np.arange(-90,  90, 1.0)
LON_OUT = np.arange(  1, 361, 1.0)

# Save directory
SAVE_DIR = "/glade/work/diegovar/climatologiesthesis/"
os.makedirs(SAVE_DIR, exist_ok=True)

# Catalog path
CATALOG_URL = (
    "/glade/collections/cmip/catalog/intake-esm-datastore/"
    "catalogs/glade-cmip6.json"
)

LOCAL_DATA_DIR = "/glade/derecho/scratch/diegovar/cmip6"


# =============================================================================
# MODEL → MEMBER ID MAPPING
# Each model is assigned its center-designated standard member ID.
# f2 is used for models where f1 has known forcing errors or is non-standard.
# Source: available_cmip6_data-3.xlsx + Kwiatkowski et al. (2020) conventions.
# =============================================================================

MODEL_MEMBERS = {
    # ── Core Si* ensemble ─────────────────────────────────────────────────────
    
    "CESM2":              "r1i1p1f1",
    #"CESM2-FV2":          "r1i1p1f1",   # same MARBL-BEC as CESM2; variant only
    "CESM2-WACCM":        "r1i1p1f1",   # same MARBL-BEC as CESM2; variant only
    #"CESM2-WACCM-FV2":    "r1i1p1f1",   # same MARBL-BEC as CESM2; variant only
    "GFDL-ESM4":          "r1i1p1f1",
    "IPSL-CM6A-LR":       "r1i1p1f1",
    "UKESM1-0-LL":        "r1i1p1f2",   # f2 required: f1 has aerosol forcing error
    "CNRM-ESM2-1":        "r1i1p1f2",   # f2 required: center-designated standard run
    # ── Partial Si* ───────────────────────────────────────────────────────────
    "GISS-E2-1-G-CC":     "r1i1p1f1",   # CC variant; si & no3 confirmed available
    "NorESM2-LM":         "r1i1p1f1",   # si & no3 present; missing phydiat, chl
    "NorCPM1":            "r1i1p1f1",   # decadal model; si & no3 present; limited vars
    # ── Physical variables only (no si/no3 in available files) ───────────────
    "GFDL-CM4":           "r1i1p1f1",
    "MIROC-ES2L":         "r1i1p1f2",   # f2 confirmed; si/no3 unavailable on NCAR
    "MPI-ESM1-2-HR":      "r1i1p1f1",
    #"MPI-ESM1-2-LR":      "r1i1p1f1",
    #"MPI-ESM-1-2-HAM":    "r1i1p1f1",
    "NorESM2-MM":         "r1i1p1f1",
    "ACCESS-ESM1-5":      "r1i1p1f1",
    "CanESM5":            "r1i1p1f1",
    # ── Excluded (insufficient variable coverage) ─────────────────────────────
 #   "GISS-E2-1-G":        "r1i1p1f1",   # only tos, mlotst, so available
 #   "IPSL-CM5A2-INCA":    "r1i1p1f1",   # only so available; CMIP5-era variant
 #   "IPSL-CM6A-LR-INCA":  "r1i1p1f1",   # only so available; exclude from analysis
}

# MODELS list derived from MODEL_MEMBERS so the two stay in sync automatically
MODELS = list(MODEL_MEMBERS.keys())

VARIABLES = ['si', 'no3']#"no3", "si", 'si*', 'dfe']#, 'po4', 'tos', 'bsi', 'phyc', 'phydiaz', 'phydiat', 'mlost', 'tos', 'phydiat_r', 'si*', 'intpp' ] #tos", "phyc","si","no3","phydiat","phydiaz","bsi","phydiat_r", "si*"]

FRONT_ZONE_DIR = "/glade/u/home/diegovar/teaching/Tutorials/Fronts/"

FRONT_ZONE_FILES = {
    "siz": "siz_grid.npy",
    "az":  "az_grid.npy",
    "saz": "saz_grid.npy",
    "stz": "stz_grid.npy",
    "pfz": "pfz_grid.npy",
}

ACTIVE_ZONES = None   # None = use all zones defined above

# Grid labels to try in order (member ID is now determined per-model above)
GRID_LIST = ["gr", "gn"]

# Unit-conversion functions applied AFTER standardize_coords
# Each function receives an xr.DataArray and returns a transformed xr.DataArray
UNIT_CONVERSIONS = {
    "phyc":    lambda da: da * 12 * 1000,          # molC/m³ → mgC/m³
    "phydiat": lambda da: da * 12 * 1000,
    "phydiaz": lambda da: da * 12 * 1000,
    "zooc":    lambda da: da * 12 * 1000,
    "zoocos":  lambda da: da * 12 * 1000,
    "chl":     lambda da: da * 1e6,                # kg/m³  → mg/m³
    "intpp":   lambda da: da * 60*60*24*365*12,    # molC/m²/s → gC/m²/yr
    "epc100":  lambda da: da * 31_536_000,
    "fgco2":   lambda da: da * 1000*3600*24*365 / (44/12),
    "spco2":   lambda da: da * 1000 / 101.325,
    "dfe":     lambda da: da * 1e6,
    "no3":     lambda da: da * 1e3,
    "si":      lambda da: da * 1e3,
}

# Variables that must be clipped to >= 0 before processing
NON_NEGATIVE_VARS = {"phyc", "chl", "no3", "si", "dfe", "dissic", "talk"}

SKIP_COMBOS = {
    ("CNRM-ESM2-1", "phydiat_r"),
}


In [11]:
# =============================================================================
# DASK CLUSTER
# =============================================================================
# 4 nodes × 32 GB = 128 GB total.
# 8 workers × 2 threads × 4 GB = 64 GB active; leaves headroom for OS/IO.
# Increase n_workers to 12 if jobs are consistently fast and memory is free.

def start_cluster():
    cluster = LocalCluster(
        n_workers=6,           # fewer workers = more memory each
        threads_per_worker=2,
        memory_limit="5.5GB",    # 4 × 7GB = 28GB, leaves headroom for OS
        dashboard_address=":34365",
    )
    
    client = Client(cluster)
    print(client)
    return client

In [12]:
# =============================================================================
# CATALOG HELPER
# =============================================================================

# Cache the catalog so we only parse the JSON once per session
_CATALOG_CACHE = None

def _get_catalog():
    global _CATALOG_CACHE
    if _CATALOG_CACHE is None:
        _CATALOG_CACHE = intake.open_esm_datastore(CATALOG_URL)
    return _CATALOG_CACHE


# =============================================================================
# CATALOG / LOCAL PATH RESOLUTION
# =============================================================================

def get_path(model, variable, data_type, experiment_id, activity_id):
    """
    Return file paths for the correct member_id / grid_label combo.

    The member ID is looked up from MODEL_MEMBERS — each model has one
    designated standard run based on modeling-center recommendations and
    confirmed data availability (see config cell).

    Search order:
      1. GLADE intake-ESM catalog  (primary)
      2. Local directory at LOCAL_DATA_DIR  (fallback)
    """
    # ------------------------------------------------------------------ #
    # 1. Catalog search — use the single correct member ID for this model  #
    # ------------------------------------------------------------------ #
    cat = _get_catalog()
    run = MODEL_MEMBERS.get(model, "r1i1p1f1")  # safe default if model not in dict
    print(f"    [member] {model} → {run}")

    for grid in GRID_LIST:
        subset_grid = cat.search(
            experiment_id=[experiment_id],
            table_id=data_type,
            variable_id=variable,
            source_id=model,
            member_id=run,
            activity_id=activity_id,
            grid_label=grid,
        )
        if not subset_grid.df.empty:
            return subset_grid.df["path"].tolist()

    # ------------------------------------------------------------------ #
    # 2. Local-directory fallback                                          #
    # ------------------------------------------------------------------ #
    return get_path_local(model, variable, data_type, experiment_id)


def get_path_local(model, variable, data_type, experiment_id):
    """
    Search LOCAL_DATA_DIR for NetCDF files that match the requested
    model / variable / experiment combination.

    Tries two common CMIP6 sub-directory layouts in order:

      Layout A (DRS-style, most common on GLADE scratch):
        <root>/<activity>/<institution>/<model>/<experiment>/
               <member>/<table>/<variable>/<grid>/<version>/*.nc

      Layout B (flatter, often used for personal downloads):
        <root>/<model>/<variable>/**/*.nc
        <root>/<variable>/<model>/**/*.nc

    Returns a sorted list of matching paths, or [] if nothing is found.
    """
    import glob

    # ---- Layout A: walk the full DRS tree under LOCAL_DATA_DIR ----------
    #  We use wildcards for the parts we don't know (institution, member,
    #  grid, version) but pin the pieces we do know.
    pattern_A = os.path.join(
        LOCAL_DATA_DIR,
        "*",            # activity_id  (e.g. CMIP, DCPP …)
        "*",            # institution_id
        model,
        experiment_id,
        "*",            # member_id
        data_type,
        variable,
        "*",            # grid_label
        "*",            # version
        "*.nc",
    )

    # ---- Layout B-1: <root>/<model>/<variable>/**/*.nc ------------------
    pattern_B1 = os.path.join(LOCAL_DATA_DIR, model, variable, "**", "*.nc")

    # ---- Layout B-2: <root>/<variable>/<model>/**/*.nc ------------------
    pattern_B2 = os.path.join(LOCAL_DATA_DIR, variable, model, "**", "*.nc")

    # ---- Layout B-3: flat, anything with model+variable in the path -----
    #  Last-resort: any .nc file whose path contains both strings.
    pattern_B3 = os.path.join(LOCAL_DATA_DIR, "**", "*.nc")

    for pattern in (pattern_A, pattern_B1, pattern_B2):
        hits = sorted(glob.glob(pattern, recursive=True))
        if hits:
            print(f"    [local fallback] found {len(hits)} file(s) via: {pattern}")
            return hits

    # Last resort: broad search filtered by name fragments
    all_nc = glob.glob(pattern_B3, recursive=True)
    hits = [
        p for p in all_nc
        if model in p and variable in p and experiment_id in p
    ]
    if hits:
        print(f"    [local fallback – broad search] found {len(hits)} file(s)")
        return sorted(hits)

    return []   # truly nothing found

# =============================================================================
# COORDINATE / DEPTH HELPERS
# =============================================================================

def standardize_coords(ds, depth_level):
    """Rename non-standard coordinate names and select the requested depth."""
    if "nav_lat" in ds.coords:
        ds = ds.rename({"nav_lon": "lon", "nav_lat": "lat"})
    if "latitude" in ds.coords and "lat" not in ds.coords:
        ds = ds.rename({"longitude": "lon", "latitude": "lat"})
    for depth_dim in ("lev_partial", "olevel", "lev"):
        if depth_dim in ds.coords:
            ds = ds.sel({depth_dim: depth_level}, method="nearest")
            break
    return ds


# =============================================================================
# REGRIDDER CACHE
# =============================================================================
# Reusing regridders across variables for the same model saves significant time
# because building the bilinear weight matrix is expensive.

_REGRIDDER_CACHE: dict = {}

def get_regridder(ds_data, ds_target):
    """Return a cached xe.Regridder; create one if the grid is new.
    
    Handles both rectilinear (1-D lat/lon) and curvilinear (2-D lat/lon)
    grids by using grid shape and corner values as a hashable cache key.
    """
    lat_vals = ds_data.lat.values
    lon_vals = ds_data.lon.values

    # Flatten to 1-D for consistent key construction regardless of grid type
    lat_flat = lat_vals.ravel()
    lon_flat = lon_vals.ravel()

    key = (
        lat_vals.shape,
        lon_vals.shape,
        float(lat_flat[0]),
        float(lat_flat[-1]),
        float(lon_flat[0]),
        float(lon_flat[-1]),
    )

    if key not in _REGRIDDER_CACHE:
        ds_in = xr.Dataset({"lat": ds_data.lat, "lon": ds_data.lon})
        _REGRIDDER_CACHE[key] = xe.Regridder(
            ds_in, ds_target, "bilinear",
            periodic=True, ignore_degenerate=True,
        )
    return _REGRIDDER_CACHE[key]


# =============================================================================
# DATA LOADING HELPERS
# =============================================================================

OPEN_KWARGS = dict(
    chunks={"time": 24},   # spatial dims vary by model; chunk time only
    parallel=True,
    combine="by_coords",
    data_vars="minimal",   # avoids FutureWarning + faster combine
    coords="minimal",
    compat="override",
)


def _open_var(model, var, data_type="Omon",
              experiment_id="historical", activity_id="CMIP"):
    """Open a single CMIP6 variable as a lazy xarray DataArray."""
    path = get_path(model, var, data_type, experiment_id, activity_id)
    if not path:
        return None
    ds = xr.open_mfdataset(path, **OPEN_KWARGS)
    ds = ds.sel(time=slice(START_YEAR, END_YEAR))
    ds = standardize_coords(ds, DEPTH_LEVEL)
    return ds[var]

def load_variable(model, var):
    """
    Load and preprocess a DataArray for *var* from *model*.
    Returns (DataArray, var_name) or (None, None) if data unavailable.
    Handles derived variables (taum, phydiat_r, si*) transparently.
    """
    if (model, var) in SKIP_COMBOS:
        print(f"    Skipping known missing combo: {model} / {var}")
        return None, None

    # ------------------------------------------------------------------ taum
    if var == "taum":
        da1 = _open_var(model, "tauuo")
        da2 = _open_var(model, "tauvo")
        if da1 is None or da2 is None:
            return None, None
        da = xr.apply_ufunc(
            np.sqrt, da1**2 + da2**2,
            dask="parallelized", output_dtypes=[da1.dtype],
        )
        da = da.assign_coords(lat=da1.lat, lon=da1.lon)
        return da, "taum"

    # ------------------------------------------------------------ phydiat_r
    if var == "phydiat_r":
        da1 = _open_var(model, "phydiat")
        da2 = _open_var(model, "phyc")
        if da1 is None or da2 is None:
            return None, None
        return da1 / da2, "phydiat_r"

    # ------------------------------------------------------------------ si*
    if var == "si*":
        da1 = _open_var(model, "si")
        da2 = _open_var(model, "no3")
        if da1 is None or da2 is None:
            return None, None
        da1 = da1.where(da1 >= 0)   # clip unphysical negatives before subtracting
        da2 = da2.where(da2 >= 0)
        return da1 * 1e3 - da2 * 1e3, "si*"

    # --------------------------------------------------------- standard var
    da = _open_var(model, var)
    if da is None:
        return None, None

    # Clip non-physical negatives
    if var in NON_NEGATIVE_VARS:
        da = da.where(da >= 0)

    # Unit conversion
    if var in UNIT_CONVERSIONS:
        da = UNIT_CONVERSIONS[var](da)

    return da, var

def load_front_zones():
    """
    Load all zone masks defined in FRONT_ZONE_FILES and return them as a dict
    of xr.DataArray objects on the same 1° lat/lon grid used for regridding.

    The raw .npy files follow the FrontSeparation convention:
      - shape  : (n_lat, n_lon)
      - lat    : np.linspace(90, -90, n_lat)   [north → south]
      - lon    : np.linspace(-180, 180, n_lon)
      - values : 1 inside zone, 0 outside (converted to NaN outside)

    After loading, each mask is regridded/reindexed onto the 1° output grid
    (LAT_OUT / LON_OUT) so it aligns with the regridded CMIP6 data.
    Returns a dict {zone_name: xr.DataArray(lat, lon)} with 1 inside / NaN outside.
    """
    active = ACTIVE_ZONES if ACTIVE_ZONES is not None else list(FRONT_ZONE_FILES.keys())
    zones  = {}

    for name in active:
        fpath = os.path.join(FRONT_ZONE_DIR, FRONT_ZONE_FILES[name])
        raw   = np.load(fpath).astype(float)           # (n_lat, n_lon)
        n_lat, n_lon = raw.shape

        # Reconstruct the native lat/lon matching FrontSeparation convention
        lat_native = np.linspace(90, -90, n_lat)        # N → S
        lon_native = np.linspace(-180, 180, n_lon + 1)[:-1]  # exclude duplicate +180 endpoint

        # 0 → NaN  (so multiplying a DataArray keeps only zone pixels)
        raw = np.where(raw == 0, np.nan, raw)

        da_raw = xr.DataArray(
            raw,
            coords={"lat": lat_native, "lon": lon_native},
            dims=["lat", "lon"],
            name=name,
        )

        # --- Flip to S→N so lat is monotonically increasing (required for sel/interp) ---
        da_raw = da_raw.sortby("lat")

        # --- Convert lon from [-180,180] to [1,361] to match LON_OUT ---
        lon_shifted = ((da_raw.lon.values + 180) % 360) + 1
        da_raw = da_raw.assign_coords(lon=lon_shifted).sortby("lon")

        # --- Nearest-neighbour reindex onto the 1° output grid ---
        da_regrid = da_raw.interp(
            lat=LAT_OUT, lon=LON_OUT, method="nearest",
            kwargs={"fill_value": np.nan},
        )

        zones[name] = da_regrid
        print(f"  Loaded zone mask: {name}  "
              f"(non-NaN cells: {int(da_regrid.notnull().sum().values)})")

    return zones

In [13]:
# =============================================================================
# CLIMATOLOGY COMPUTATION  (per front zone)
# =============================================================================

def compute_climatology(da, var, ds_grid, zone_mask):
    """
    Given a lazy DataArray *da* on its native grid and a 2-D zone mask
    (1 inside zone / NaN outside, on the 1° output grid):

      1. Regrid *da* to the 1° target grid
      2. Apply the zone mask  (pixels outside become NaN)
      3. Compute time mean over the masked region
      4. Compute anomaly
      5. Detrend anomaly
      6. Compute detrended monthly climatology (area-average within zone)

    Returns a numpy array of shape (12,).
    """
    # --- 1. Regrid ---
    regridder = get_regridder(da, ds_grid)
    da_rg     = regridder(da)                          # still lazy

    # --- 2. Apply zone mask (broadcast over time automatically) ---
    da_masked = da_rg * zone_mask     # NaN outside zone; values inside preserved

    # --- 3. Time mean and anomaly (lazy) ---
    time_mean = da_masked.mean(dim="time", skipna=True)
    anom      = da_masked - time_mean

    # --- 4. Detrend ---
    anom_filled = anom.fillna(0).chunk({"time": -1, "lat": 45, "lon": 45})  # time must be unchunked for detrend(axis=0)
    anom_detrended = xr.apply_ufunc(
        detrend,
        anom_filled,
        kwargs={"axis": 0},
        dask="parallelized",
        output_dtypes=[anom.dtype],
    ).where(~anom.isnull())

    detrended = anom_detrended + time_mean

    # --- 5. Monthly climatology — area-average within zone, single compute ---
    clim = (
        detrended
        .groupby("time.month")
        .mean("time", skipna=True)
        .mean("lon",  skipna=True)
        .mean("lat",  skipna=True)
        .compute()
        .values
    )
    return clim          # shape (12,)


# =============================================================================
# SAVE / LOAD HELPERS
# =============================================================================

def save_path(model, var, zone_name):
    """Output file path encodes model, variable, zone, depth, and year range."""
    suffix   = (f"_{DEPTH_LEVEL}m" if DEPTH_LEVEL != 0 else "")
    safe_var = var.replace("*", "star")  # avoid shell-unsafe * in filename
    fname    = f"{model}_{safe_var}{suffix}_{zone_name}_{START_YEAR}{END_YEAR}.txt"
    return os.path.join(SAVE_DIR, fname)


def save_climatology(clim_values, model, var, zone_name):
    path = save_path(model, var, zone_name)
    with open(path, "w") as f:
        for v in clim_values:
            f.write(f"{v}\n")
    print(f"    Saved → {path}")


# =============================================================================
# MAIN LOOP
# =============================================================================

def run():
    # 1° output grid (defined once)
    ds_grid = xr.Dataset({
        "lat": (["lat"], LAT_OUT),
        "lon": (["lon"], LON_OUT),
    })

    # Load all front-zone masks once (they are small 2-D arrays)
    print("Loading front zone masks...")
    front_zones = load_front_zones()
    print(f"Zones active: {list(front_zones.keys())}\n")

    for model in MODELS:
        print(f"\n{'='*60}")
        print(f"Model: {model}")
        print(f"{'='*60}")

        for var in VARIABLES:
            print(f"  Variable: {var}")

            da = None
            try:
                # --- Load once (lazy) — reuse across all zones ---
                da, var_name = load_variable(model, var)
                if da is None:
                    print("    No data found — skipping.")
                    continue

                for zone_name, zone_mask in front_zones.items():
                    out_path = save_path(model, var, zone_name)
                    if os.path.exists(out_path) and not OVERWRITE:
                        print(f"    [{zone_name}] Output exists — skipping.")
                        continue
                        
                    if os.path.exists(out_path) and OVERWRITE:
                        print(f"    [{zone_name}] Output exists — overwriting (OVERWRITE=True).")
    
                    print(f"    [{zone_name}] Computing...")
                    clim = compute_climatology(da, var_name, ds_grid, zone_mask)
                    save_climatology(clim, model, var, zone_name)

            except Exception as e:
                print(f"    ERROR for {model}/{var}: {e}")

            finally:
                del da
                gc.collect()
                client.run(gc.collect)
                client.run(lambda: None)

    print("\nAll done.")

In [ ]:
# =============================================================================
# ENTRY POINT
# =============================================================================
OVERWRITE = False

if __name__ == "__main__":
    client = start_cluster()
    client
    print(client)
    try:
        run()
    finally:
        client.close()

<Client: 'tcp://127.0.0.1:39075' processes=6 threads=12, memory=30.73 GiB>
<Client: 'tcp://127.0.0.1:39075' processes=6 threads=12, memory=30.73 GiB>
Loading front zone masks...
  Loaded zone mask: siz  (non-NaN cells: 10328)
  Loaded zone mask: az  (non-NaN cells: 1518)
  Loaded zone mask: saz  (non-NaN cells: 2141)
  Loaded zone mask: stz  (non-NaN cells: 3903)
  Loaded zone mask: pfz  (non-NaN cells: 1600)
Zones active: ['siz', 'az', 'saz', 'stz', 'pfz']


Model: CESM2
  Variable: si
    [member] CESM2 → r1i1p1f1
